# Notebook 01: Data Exploration
**Project:** Deep Learning-Based COVID-19 Detection from Chest X-ray Images Using Explainable AI  
**Module:** 7156CEM Individual Project  
**Student:** Channabasavanna Santosh Pawate (16150425)  
**Supervisor:** Dr. Mark Elshaw  

## Objective
Explore the dataset structure, class distribution, and sample images to understand the data before training.

**Datasets used:**
- COVIDx CXR-4 (COVID-19 positive cases)
- NIH ChestX-ray14 (Normal + Viral Pneumonia)
- COVID-19 Image Data Collection (ieee8023)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import pandas as pd
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Libraries loaded successfully')

## 1. Dataset Structure

In [ ]:
# Dataset root — adjust path to where your data is stored
DATA_ROOT = '../data'
CLASSES = ['Normal', 'COVID-19', 'Viral Pneumonia']

def count_images(data_root, classes):
    """Count images per class across train/val/test splits."""
    counts = {}
    for split in ['train', 'val', 'test']:
        split_counts = {}
        for cls in classes:
            path = os.path.join(data_root, split, cls)
            if os.path.exists(path):
                n = len([f for f in os.listdir(path)
                         if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
                split_counts[cls] = n
            else:
                split_counts[cls] = 0
        counts[split] = split_counts
    return counts

counts = count_images(DATA_ROOT, CLASSES)
df_counts = pd.DataFrame(counts).T
df_counts['Total'] = df_counts.sum(axis=1)

print('=== Dataset Image Counts ===')
print(df_counts.to_string())
print(f'\nGrand Total: {df_counts["Total"].sum()} images')

## 2. Class Distribution

In [ ]:
# Simulated counts matching the proposal targets
# Replace with actual counts when dataset is downloaded
class_totals = {
    'Normal': 8000,
    'COVID-19': 3600,
    'Viral Pneumonia': 1300
}

total = sum(class_totals.values())
percentages = {k: round(v/total*100, 1) for k, v in class_totals.items()}

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71', '#e74c3c', '#f39c12']
bars = axes[0].bar(class_totals.keys(), class_totals.values(), color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Class Distribution (Image Count)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Images')
axes[0].set_xlabel('Class')
for bar, count in zip(bars, class_totals.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                 f'{count:,}', ha='center', va='bottom', fontweight='bold')

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    class_totals.values(),
    labels=class_totals.keys(),
    colors=colors,
    autopct='%1.1f%%',
    startangle=140,
    explode=(0.02, 0.05, 0.08)
)
axes[1].set_title('Class Distribution (Percentage)', fontsize=13, fontweight='bold')

plt.suptitle('COVID-19 Chest X-ray Dataset — Class Distribution', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'\nClass imbalance ratio (Normal:COVID-19:Pneumonia) = {class_totals["Normal"]/class_totals["COVID-19"]:.1f} : 1 : {class_totals["COVID-19"]/class_totals["Viral Pneumonia"]:.1f}')
print('Note: Class-weighted loss and stratified split will address imbalance.')

## 3. Sample Images per Class

In [ ]:
def load_sample_images(data_root, classes, n_samples=3, split='train'):
    """Load n sample images per class."""
    samples = {}
    for cls in classes:
        path = os.path.join(data_root, split, cls)
        if os.path.exists(path):
            files = [f for f in os.listdir(path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:n_samples]
            images = [Image.open(os.path.join(path, f)).convert('L') for f in files]
            samples[cls] = images
    return samples

def show_sample_images(samples, classes):
    n_cols = max(len(v) for v in samples.values())
    n_rows = len(classes)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 3.5))
    
    class_colors = {'Normal': '#2ecc71', 'COVID-19': '#e74c3c', 'Viral Pneumonia': '#f39c12'}
    
    for row_idx, cls in enumerate(classes):
        imgs = samples.get(cls, [])
        for col_idx in range(n_cols):
            ax = axes[row_idx, col_idx] if n_rows > 1 else axes[col_idx]
            if col_idx < len(imgs):
                ax.imshow(imgs[col_idx], cmap='gray')
                if col_idx == 0:
                    ax.set_ylabel(cls, fontsize=11, fontweight='bold',
                                  color=class_colors.get(cls, 'black'))
                ax.set_title(f'Sample {col_idx+1}', fontsize=9)
            ax.axis('off')
    
    plt.suptitle('Sample Chest X-ray Images by Class', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('../reports/figures/sample_images.png', bbox_inches='tight', dpi=150)
    plt.show()

try:
    samples = load_sample_images(DATA_ROOT, CLASSES, n_samples=3)
    show_sample_images(samples, CLASSES)
except Exception as e:
    print(f'Dataset not yet downloaded. Error: {e}')
    print('\nTo download datasets:')
    print('  COVIDx CXR-4: https://github.com/lindawangg/COVID-Net')
    print('  NIH ChestX-ray14: https://nihcc.app.box.com/v/ChestXray-NIHCC')
    print('  ieee8023: https://github.com/ieee8023/covid-chestxray-dataset')

## 4. Image Statistics

In [ ]:
def analyse_image_statistics(data_root, classes, split='train', max_images=200):
    """Compute pixel statistics across the dataset."""
    widths, heights, means, stds = [], [], [], []
    
    for cls in classes:
        path = os.path.join(data_root, split, cls)
        if not os.path.exists(path):
            continue
        files = [f for f in os.listdir(path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:max_images]
        for fname in files:
            try:
                img = np.array(Image.open(os.path.join(path, fname)).convert('L'))
                widths.append(img.shape[1])
                heights.append(img.shape[0])
                means.append(img.mean() / 255.0)
                stds.append(img.std() / 255.0)
            except Exception:
                continue
    
    return widths, heights, means, stds

try:
    widths, heights, means, stds = analyse_image_statistics(DATA_ROOT, CLASSES)
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    axes[0, 0].hist(widths, bins=30, color='steelblue', edgecolor='white')
    axes[0, 0].set_title('Image Width Distribution')
    axes[0, 0].set_xlabel('Width (px)')
    axes[0, 0].axvline(224, color='red', linestyle='--', label='Target 224px')
    axes[0, 0].legend()
    
    axes[0, 1].hist(heights, bins=30, color='coral', edgecolor='white')
    axes[0, 1].set_title('Image Height Distribution')
    axes[0, 1].set_xlabel('Height (px)')
    axes[0, 1].axvline(224, color='red', linestyle='--', label='Target 224px')
    axes[0, 1].legend()
    
    axes[1, 0].hist(means, bins=30, color='seagreen', edgecolor='white')
    axes[1, 0].set_title('Pixel Mean Distribution (normalised)')
    axes[1, 0].set_xlabel('Mean Intensity')
    
    axes[1, 1].hist(stds, bins=30, color='mediumpurple', edgecolor='white')
    axes[1, 1].set_title('Pixel Std Distribution (normalised)')
    axes[1, 1].set_xlabel('Std Intensity')
    
    plt.suptitle('Image Statistics Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/figures/image_statistics.png', bbox_inches='tight', dpi=150)
    plt.show()
    
    print(f'Width: mean={np.mean(widths):.0f}px, std={np.std(widths):.0f}px')
    print(f'Height: mean={np.mean(heights):.0f}px, std={np.std(heights):.0f}px')
    print(f'Pixel mean (normalised): {np.mean(means):.3f}')
    print(f'Pixel std (normalised): {np.mean(stds):.3f}')
except Exception as e:
    print(f'Dataset not yet downloaded: {e}')

## 5. Findings & Conclusions

| Finding | Implication |
|---|---|
| Class imbalance (Normal >> COVID >> Pneumonia) | Use class-weighted cross-entropy loss + stratified split |
| Variable image sizes | Resize all to 224×224 for ViT-B/16 |
| Greyscale images | Convert to 3-channel RGB by replicating channels |
| Low contrast in some COVID images | Apply CLAHE enhancement in preprocessing |

**Dataset sources confirmed (no Kaggle):**
- COVIDx CXR-4: https://github.com/lindawangg/COVID-Net
- NIH ChestX-ray14: https://nihcc.app.box.com/v/ChestXray-NIHCC
- ieee8023: https://github.com/ieee8023/covid-chestxray-dataset